### Collecting financial S&P 500 stock market related headlines with Marketaux API


Marketaux offers a 100% free plan with no payment details required, tracking over 80 global markets with news from more than 5,000 sources in 30+ languages. Marketaux You can filter by the ^GSPC or SPX entity. <br>The response naturally maps to your schema:
title       → headline
published_at → date
source      → source

In [18]:
API_KEY = "mplaJ8KEt5IuVdvND1nbXZNPoiiMIL39nLgMB31e"

In [21]:
# Test multiple approaches at once
tests = [
    {'symbols': 'SPY'},                      # S&P 500 ETF
    {'symbols': 'VOO'},                      # Vanguard S&P 500 ETF
    {'search': 'S&P 500'},                   # keyword search
    {'search': 'SPX'},                       # keyword search
]

base_params = {
    'api_token': API_KEY,
    'published_after': '2026-02-01T00:00:00',
    'published_before': '2026-02-28T23:59:59',
    'limit': 3,
    'language': 'en',
}

for test in tests:
    params = {**base_params, **test}
    response = requests.get(
        'https://api.marketaux.com/v1/news/all',
        params=params,
        verify=certifi.where()
    )
    meta = response.json().get('meta', {})
    print(f"{test} → found: {meta.get('found')}, returned: {meta.get('returned')}")


{'symbols': 'SPY'} → found: 83, returned: 3
{'symbols': 'VOO'} → found: 31, returned: 3
{'search': 'S&P 500'} → found: 5259, returned: 3
{'search': 'SPX'} → found: 534, returned: 3


In [22]:
# Python 3
# import http.client
import requests
import certifi
import urllib.parse
import json
import time

spx_articles = []
page = 1

while True:
    params = {
        'api_token': API_KEY,
        'search': 'S&P 500',
        'published_after':  '2026-02-01T00:00:00',
        'published_before': '2026-02-28T23:59:59',
        'limit': 50,
        'page': page,
        'language': 'en',
    }
    response = requests.get(
        'https://api.marketaux.com/v1/news/all',
        params=params,
        verify=certifi.where()
    )
    body = response.json()

    # Catch API errors (quota exceeded, auth, etc.)
    if 'error' in body:
        print(f"API error on page {page}: {body['error']}")
        break

    articles = body.get('data', [])
    if not articles:
        print(f"No more articles returned on page {page}. Done.")
        break

    spx_articles.extend(articles)
    total_found = body['meta']['found']
    print(f"Page {page}: fetched {len(articles)} | collected {len(spx_articles)} / {total_found}")

    # Stop if we've collected everything
    if len(spx_articles) >= total_found:
        break

    time.sleep(1)
    page += 1

# Normalize
rows = []
for a in spx_articles:
    rows.append({
        'headline': a.get('title'),
        'date':     a.get('published_at'),
        'source':   a.get('source'),
    })

# Preview
for row in rows[:5]:
    print(row)
print(f"\nTotal articles collected: {len(rows)}")

# Save to CSV
with open('spx_headlines_feb2026.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['headline', 'date', 'source'])
    writer.writeheader()
    writer.writerows(rows)
print("Saved to spx_headlines_feb2026.csv")


Page 1: fetched 3 | collected 3 / 5259
Page 2: fetched 3 | collected 6 / 5259
Page 3: fetched 3 | collected 9 / 5259
Page 4: fetched 3 | collected 12 / 5259
Page 5: fetched 3 | collected 15 / 5259
Page 6: fetched 3 | collected 18 / 5259
Page 7: fetched 3 | collected 21 / 5259
Page 8: fetched 3 | collected 24 / 5259
Page 9: fetched 3 | collected 27 / 5259
Page 10: fetched 3 | collected 30 / 5259
Page 11: fetched 3 | collected 33 / 5259
Page 12: fetched 3 | collected 36 / 5259
Page 13: fetched 3 | collected 39 / 5259
Page 14: fetched 3 | collected 42 / 5259
Page 15: fetched 3 | collected 45 / 5259
Page 16: fetched 3 | collected 48 / 5259
Page 17: fetched 3 | collected 51 / 5259
Page 18: fetched 3 | collected 54 / 5259
Page 19: fetched 3 | collected 57 / 5259
Page 20: fetched 3 | collected 60 / 5259
Page 21: fetched 3 | collected 63 / 5259
Page 22: fetched 3 | collected 66 / 5259
Page 23: fetched 3 | collected 69 / 5259
Page 24: fetched 3 | collected 72 / 5259
Page 25: fetched 3 | collect

NameError: name 'csv' is not defined

In [24]:
import csv

with open('spx_headlines_feb2026.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['headline', 'date', 'source'])
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} rows to spx_headlines_feb2026.csv")

Saved 276 rows to spx_headlines_feb2026.csv


In [23]:
rows

[{'headline': 'S&P 500 Snapshot: Second Straight Loss',
  'date': '2026-02-17T16:20:11.000000Z',
  'source': 'etftrends.com'},
 {'headline': 'S&P 500 Snapshot: Best Day Since May',
  'date': '2026-02-06T23:18:33.000000Z',
  'source': 'etftrends.com'},
 {'headline': 'The Best S&P 500 ETF to Invest $500 in Right Now',
  'date': '2026-02-14T12:05:00.000000Z',
  'source': 'finance.yahoo.com'},
 {'headline': 'The Outlook for S&P 500 Dividends in February 2026',
  'date': '2026-02-27T09:00:15.000000Z',
  'source': 'realclearmarkets.com'},
 {'headline': 'The S&P 500 Is Stuck. What History Says Happens Next.',
  'date': '2026-02-22T10:04:00.000000Z',
  'source': 'finance.yahoo.com'},
 {'headline': "Forget Invesco's S&P 500 ETF and Buy This Instead",
  'date': '2026-02-13T15:50:00.000000Z',
  'source': 'yahoo.com'},
 {'headline': 'S&P 500 Earnings: 2026 Growth Estimates Face Early Downward Pressure',
  'date': '2026-02-02T06:50:00.000000Z',
  'source': 'investing.com'},
 {'headline': 'Is the Va